In [ ]:
import pandas as pd
import numpy as np
import h3
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from pathlib import Path

from sklearn.svm import SVR, LinearSVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
print("Loading final grid dataset...")
final_grid = pd.read_parquet("../data/final_grid.parquet")
print(f"Final grid loaded. Shape: {final_grid.shape}")

final_grid.head()

In [ ]:
# Calculate distance to Chicago Loop (downtown center)
def haversine_distance(lat1, lon1, lat2=41.8781, lon2=-87.6298):
    r = 6371 # earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    a = np.sin(delta_phi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c

In [ ]:
final_grid['distance_to_loop'] = haversine_distance(final_grid['latitude'], final_grid['longitude'])
print(final_grid[['h3_index', 'latitude', 'longitude', 'distance_to_loop']].head(2))

In [ ]:
# Calendar features
final_grid['month'] = final_grid['hour'].dt.month
final_grid['day_of_week'] = final_grid['hour'].dt.weekday
final_grid['hour_of_day'] = final_grid['hour'].dt.hour
final_grid['is_weekend'] = (final_grid['day_of_week'] >= 5).astype(int)

# Cyclic temporal features
final_grid['hour_sin'] = np.sin(2 * np.pi * final_grid['hour_of_day'] / 24)
final_grid['hour_cos'] = np.cos(2 * np.pi * final_grid['hour_of_day'] / 24)
final_grid['month_sin'] = np.sin(2 * np.pi * final_grid['month'] / 12)
final_grid['month_cos'] = np.cos(2 * np.pi * final_grid['month'] / 12)

# Holidays feature
us_holidays = holidays.US(state='IL')
final_grid['is_holiday'] = final_grid['hour'].dt.date.isin(us_holidays).astype(int)

# Select final features and target column
features = [
    'latitude', 'longitude', 'distance_to_loop',
    'poi_count', 'count_poi_types',
    '2m_temp_c', 'total_precip_mm', 'wind_speed', 'snow_cov', 'snow_depth',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'is_weekend', 'is_holiday'
]

# Fill spatial POI NA values with 0
final_grid['poi_count'] = final_grid['poi_count'].fillna(0)
final_grid['count_poi_types'] = final_grid['count_poi_types'].fillna(0)

print(f"Feature matrix prepared with columns: {features}")